# 4. What survives resampling

A hub that only appears in one network is not a finding. Both the hubs and the modules here
are defined by how often they come back across bootstrap networks.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.config import Config
from src.io_utils import data_path, read_matrix, read_table, results_path

cfg = Config.load("../config.json")

In [ ]:
from src.modules import (
    coassignment_matrix, consensus_modules, detect_modules, module_coherence, module_summary,
)
from src.network import symmetric_adjacency
from src.preprocessing import split_by_group
from src.stability import bootstrap_networks, hub_frequency, partition_ari, stable_hubs

expr = read_matrix(results_path("preprocessed", "tcga_expression.tsv.gz"))
meta = read_table(results_path("preprocessed", "tcga_meta.tsv"), index_col=0)
groups = split_by_group(expr, meta["subtype"], cfg.subtypes, cfg.min_subtype_n)
n_match = cfg.n_match or min(sub.shape[1] for sub in groups.values())

In [ ]:
name = sorted(groups)[0]
nets = bootstrap_networks(
    groups[name], n_boot=cfg.n_bootstrap, seed=cfg.seed, size=n_match,
    tree_method=cfg.tree_method, n_trees=cfg.n_trees,
    max_features=cfg.max_features, n_jobs=cfg.n_jobs, verbose=True,
)
len(nets)

In [ ]:
freq = hub_frequency(nets, cfg.top_edges, cfg.hub_top_k)
print(len(stable_hubs(freq, cfg.hub_stability_threshold)), "hubs above the threshold")
freq.head(15)

## Modules

Clustering one network gives a partition that moves as soon as you resample. Clustering the
co-assignment matrix instead gives modules defined by how often their genes cluster
together, and the coherence number is the module's stability.

In [ ]:
partitions = [detect_modules(symmetric_adjacency(net), cfg.min_module_size, cfg.max_modules)
              for net in nets]
print("modules per bootstrap:", [len(p) for p in partitions])
print("mean ARI between one network's partition and the rest:",
      round(float(partition_ari(partitions[0], partitions[1:]).mean()), 2))

In [ ]:
consensus = coassignment_matrix(partitions, list(expr.index))
modules = consensus_modules(consensus, cfg.min_module_size, cfg.max_modules)
coherence = module_coherence(consensus, modules)
pd.DataFrame({"size": {m: len(g) for m, g in modules.items()}, "coherence": coherence})

In [ ]:
kept = {m: g for m, g in modules.items() if coherence[m] >= cfg.module_stability_threshold}
print(len(kept), "of", len(modules), "modules go forward")
module_summary(kept, symmetric_adjacency(read_matrix(results_path("networks", f"{name}_vim.tsv.gz"))))